In [ ]:
# @title Set ONLINE = False if running locally
ONLINE = True

In [ ]:
#@title Basic Imports

import pandas as pd
import numpy as np
import math
import gdown
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
# Download training and submission data
if ONLINE:
    url = 'https://drive.google.com/file/d/1lDEtOq644qlaw_vh0i5c3ChVfrHJiuMc/view?usp=sharing'
    output = 'data_train.csv'
    gdown.download(url, output, fuzzy=True)
    url = 'https://drive.google.com/file/d/1KMiakfzTWYr4QJU9S5DC9e5KIo_8bb24/view?usp=sharing'
    output = 'sampleSubmission.csv'
    gdown.download(url, output, fuzzy=True)

In [ ]:
number_of_users, number_of_movies = (10000, 1000)
data_pd = pd.read_csv('data_train.csv')

In [ ]:
# Split the dataset into train and test

train_size = 0.9

train_pd, test_pd = train_test_split(data_pd, train_size=train_size, random_state=42)

In [ ]:

def extract_users_items_predictions(data_pd):
    users, movies = \
        [np.squeeze(arr) for arr in np.split(data_pd.Id.str.extract('r(\d+)_c(\d+)').values.astype(int) - 1, 2, axis=-1)]
    predictions = data_pd.Prediction.values
    return users, movies, predictions

train_users, train_movies, train_predictions = extract_users_items_predictions(train_pd)
# also create full matrix of observed values
data = np.full((number_of_users, number_of_movies), np.mean(train_pd.Prediction.values))
mask = np.zeros((number_of_users, number_of_movies)) # 0 -> unobserved value, 1->observed value

for user, movie, pred in zip(train_users, train_movies, train_predictions):
    #data[user - 1][movie - 1] = pred
    #mask[user - 1][movie - 1] = 1
    data[user][movie] = pred
    mask[user][movie] = 1

In [ ]:
rmse = lambda x, y: math.sqrt(mean_squared_error(x, y))

test_users, test_movies, test_predictions = extract_users_items_predictions(test_pd)

# test our predictions with the true values
def get_score(predictions, target_values=test_predictions):
    return rmse(predictions, target_values)

def extract_prediction_from_full_matrix(reconstructed_matrix, users=test_users, movies=test_movies):
    # returns predictions for the users-movies combinations specified based on a full m \times n matrix
    assert(len(users) == len(movies)), "users-movies combinations specified should have equal length"
    predictions = np.zeros(len(test_users))

    for i, (user, movie) in enumerate(zip(users, movies)):
        predictions[i] = reconstructed_matrix[user][movie]

    return predictions

In [ ]:
#@title Use GPU in colab: Runtime->Change Runtime type
#  use gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using device:', device)

In [ ]:
class NCF(nn.Module):
    def __init__(self, number_of_users: int, number_of_movies: int, users_embedding_size: int, movies_embedding_size: int):
        super().__init__()
        self.embedding_layer_users = nn.Embedding(
            number_of_users, users_embedding_size)
        self.embedding_layer_movies = nn.Embedding(
            number_of_movies, movies_embedding_size)

        self.feed_forward = nn.Sequential(
            nn.Linear(in_features=users_embedding_size +
                      movies_embedding_size, out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=32),
            nn.ReLU(),
            nn.Linear(in_features=32, out_features=16),
            nn.ReLU(),
            nn.Linear(in_features=16, out_features=1),
            nn.ReLU()
        )

    def forward(self, users, movies):
        users_embedding = self.embedding_layer_users(users)
        movies_embedding = self.embedding_layer_movies(movies)
        concat = torch.cat([users_embedding, movies_embedding], dim=1)
        return torch.squeeze(self.feed_forward(concat))

In [ ]:
# Parameters
batch_size = 128
num_epochs = 40
show_validation_score_every_epochs = 1
movies_embedding_size= 16
users_embedding_size=32
learning_rate = 1e-4
weight_decay= 3e-4

In [ ]:
def mse_loss(predictions, target):
    return torch.mean((predictions - target) ** 2)

# Build Dataloaders
train_users_torch = torch.tensor(train_users, device=device)
train_movies_torch = torch.tensor(train_movies, device=device)
train_predictions_torch = torch.tensor(train_predictions, device=device)

train_dataloader = DataLoader(
    TensorDataset(train_users_torch, train_movies_torch, train_predictions_torch),
    batch_size=batch_size)

test_users_torch = torch.tensor(test_users, device=device)
test_movies_torch = torch.tensor(test_movies, device=device)

test_dataloader = DataLoader(
    TensorDataset(test_users_torch, test_movies_torch),
    batch_size=batch_size)

ncf = NCF(number_of_users=number_of_users, number_of_movies=number_of_movies,
          users_embedding_size=users_embedding_size,movies_embedding_size=movies_embedding_size).to(device)
optimizer = optim.Adam(ncf.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3)


In [ ]:
step = 0
with tqdm(total=len(train_dataloader) * num_epochs) as pbar:
    for epoch in range(num_epochs):
        for users_batch, movies_batch, target_predictions_batch in train_dataloader:
            optimizer.zero_grad()

            predictions_batch = ncf(users_batch, movies_batch)

            loss = mse_loss(predictions_batch, target_predictions_batch)

            loss.backward()

            optimizer.step()
            pbar.update(1)
            step += 1

        if epoch % show_validation_score_every_epochs == 0:

            with torch.no_grad():
                all_predictions = []
                for users_batch, movies_batch in test_dataloader:
                    predictions_batch = ncf(users_batch, movies_batch)
                    all_predictions.append(predictions_batch)

            all_predictions = torch.cat(all_predictions)

            reconstuction_rmse = get_score(all_predictions.cpu().numpy())
            pbar.set_description('At epoch {:3d} loss is {:.4f}'.format(epoch, reconstuction_rmse))
            scheduler.step(reconstuction_rmse)
            #scheduler.step()
            print('training/test loss --->', np.sqrt(loss.item()),"/",reconstuction_rmse, "learning rate--> ",scheduler.get_last_lr()[0])



In [ ]:
# Create a new function to extract user and movie IDs from submission data
def extract_users_items(data_pd):
    users, movies = [np.squeeze(arr) for arr in np.split(data_pd.Id.str.extract('r(\d+)_c(\d+)').values.astype(int) - 1, 2, axis=-1)]
    return users, movies

# Load the submission dataset
submission_data = pd.read_csv('sampleSubmission.csv')

# Extract users and movies from the submission dataset
submission_users, submission_movies = extract_users_items(submission_data)

# Convert to torch tensors
submission_users_torch = torch.tensor(submission_users, device=device)
submission_movies_torch = torch.tensor(submission_movies, device=device)

# Create a DataLoader for the submission data
submission_dataloader = DataLoader(
    TensorDataset(submission_users_torch, submission_movies_torch),
    batch_size=batch_size
)

# Generate predictions for the submission data
ncf.eval()  # Set the model to evaluation mode
submission_predictions = []

with torch.no_grad():
    for users_batch, movies_batch in submission_dataloader:
        predictions_batch = ncf(users_batch, movies_batch)
        submission_predictions.append(predictions_batch)

submission_predictions = torch.cat(submission_predictions).cpu().numpy()

# Create a DataFrame with the required format for submission
submission_df = pd.DataFrame({
    'Id': submission_data['Id'],
    'Prediction': submission_predictions
})

# Save the DataFrame to the specified submission path
submission_df.to_csv('ncf_submission.csv', index=False)

print(f'Submission file saved')